# Corn Strategy Backtest Research

Standalone corn notebook. The strategy logic follows the same research shape as the soybean and wheat notebooks: load/confirm data, define notebook-level evaluation helpers, run generic A/B diagnostics, test corn-specific alpha sleeves/combinations, lock the final candidate, and report the final check.


In [1]:
from itertools import combinations

import pandas as pd
from IPython.display import Markdown, display

from research_config import (
    CORN_HOLDING_COST_RATE,
    CORN_TRADE_COST_PER_LOT,
    CORN_TRAIN_END,
    DEFAULT_MARGIN_PER_LOT,
    SPLIT_DATE,
)
from strategy_backtest_common import backtest_positions_with_costs, load_train_set
from grain_research import (
    build_product_flow_feature_panels,
    build_corn_product_flow_signal_universe,
    corn_signal_set_families,
    corn_average_all_signals,
    corn_equal_family_signal,
    corn_select_by_ic_signal,
    corn_trend_mr_family_signal,
    corn_dynamic_linear_family_signal,
    corn_family_signal,
    mean_product_flow_signals,
    corn_positions_from_signal,
    summarize_corn_backtest,
    build_corn_vol_regime_signal,
    corn_abundant_supply_masks,
    build_corn_carry_forward_candidates,
    make_corn_candidate,
    summarize_corn_candidates,
    run_corn_supply_guard_tests,
)


pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA_DIR = "train_set"
COMMODITY = "CORN"
TRAIN_END = pd.Timestamp(CORN_TRAIN_END)
OOS_START = pd.Timestamp(SPLIT_DATE)


## 1. Load Data And Confirm Cargill Inputs

`build_product_flow_feature_panels` is the product-flow-aligned feature builder exposed from `grain_research.py`. It uses the corn-relevant timing assumptions and includes Cargill processed/planned crush activity as a physical activity proxy for corn.


In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_product_flow_feature_panels(data)
futures_pnl = futures_pnl_all[[COMMODITY]].copy()
trading_index = futures_pnl.index

signals = build_corn_product_flow_signal_universe(feature_panels, futures_pnl_all, DATA_DIR)
families_by_set = corn_signal_set_families(signals)

display(
    pd.DataFrame(
        [
            {
                "commodity": COMMODITY,
                "start": trading_index.min().date(),
                "end": trading_index.max().date(),
                "rows": len(trading_index),
                "corn_features": feature_panels[COMMODITY].shape[1],
                "signals": len(signals),
                "has_cargill_crush_activity": {"crush_surprise", "crush_utilization"}.issubset(feature_panels[COMMODITY].columns),
                "train_rows": int((trading_index < TRAIN_END).sum()),
                "validation_rows": int(((trading_index >= TRAIN_END) & (trading_index < OOS_START)).sum()),
                "oos_rows": int((trading_index >= OOS_START).sum()),
            }
        ]
    )
)

coverage = []
for signal_set, family_map in families_by_set.items():
    for family, members in family_map.items():
        coverage.append(
            {
                "signal_set": signal_set,
                "family": family,
                "signals": len(members),
                "nonzero_signals": int(sum(series.abs().sum() > 0.0 for series in members.values())),
            }
        )
display(pd.DataFrame(coverage))


,commodity,start,end,rows,corn_features,signals,has_cargill_crush_activity,train_rows,validation_rows,oos_rows
0,CORN,2010-01-04,2020-12-31,2866,21,18,True,1565,520,781


,signal_set,family,signals,nonzero_signals
0,A,prices,7,7
1,A,fundamentals,7,7
2,A,macro,2,2
3,B,prices,7,7
4,B,fundamentals,5,5
5,alpha,eia,1,1
6,alpha,macro,2,2
7,alpha,weather,1,1


## 2. Standalone Helpers

The sibling notebooks define their evaluation helpers in one early block. Corn keeps the same notebook shape while delegating the lower-level position sizing, costs, split metrics, and guard mechanics to `grain_research.py`.


In [3]:
def evaluate_signal(test, signal_set, strategy, signal, mode="long_short", note=""):
    positions = corn_positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions_with_costs(
        positions,
        futures_pnl,
        trade_cost_per_lot=CORN_TRADE_COST_PER_LOT,
        holding_cost_rate=CORN_HOLDING_COST_RATE,
        margin_per_lot=DEFAULT_MARGIN_PER_LOT,
    )
    row = {
        "test": test,
        "signal_set": signal_set,
        "strategy": strategy,
        "mode": mode,
        "note": note,
    }
    row.update(summarize_corn_backtest(bt))
    return row, bt, positions


## 3. Generic Signal A/B Tests

Requested generic tests per signal set:

1. average all signals;
2. equal family;
3. best family by trend/MR regime;
4. dynamic linear coefficients;
5. select by IC.

Every displayed generic backtest row is `long_short` only. The volatility-regime reference is excluded from this A/B selection path.


In [4]:
generic_rows = []

for signal_set in ["A", "B"]:
    families = families_by_set[signal_set]
    trend_signal, _ = corn_trend_mr_family_signal(families, futures_pnl, feature_panels)
    dynamic_signal, _ = corn_dynamic_linear_family_signal(families, futures_pnl)
    ic_signal, _ = corn_select_by_ic_signal(families, futures_pnl)

    tests = [
        ("avg_all_signals", corn_average_all_signals(families, trading_index)),
        ("equal_family", corn_equal_family_signal(families, trading_index)),
        ("best_family_by_trend_mr", trend_signal),
        ("dynamic_linear_coeff", dynamic_signal),
        ("select_by_ic", ic_signal),
    ]
    for strategy, signal in tests:
        row, _, _ = evaluate_signal("generic", signal_set, strategy, signal, mode="long_short")
        generic_rows.append(row)

generic_results = pd.DataFrame(generic_rows).sort_values(
    ["signal_set", "validation_sharpe", "oos_sharpe"],
    ascending=[True, False, False],
)
display(generic_results[
    ["signal_set", "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "turnover"]
])


,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,turnover
2,A,best_family_by_trend_mr,long_short,0.050,1.002,-0.791,-726.143,"-1,022.804",-0.048,0.009
0,A,avg_all_signals,long_short,-0.279,0.627,0.206,146.475,-435.802,0.059,0.004
4,A,select_by_ic,long_short,0.263,0.361,-1.001,-562.334,-766.971,-0.107,0.005
1,A,equal_family,long_short,-0.164,0.340,0.665,421.742,-231.109,0.174,0.004
3,A,dynamic_linear_coeff,long_short,0.074,-1.395,-0.061,-101.698,-968.559,-0.233,0.010
7,B,best_family_by_trend_mr,long_short,-0.065,1.034,-0.761,-732.246,"-1,217.702",0.001,0.008
6,B,equal_family,long_short,-0.250,0.787,-0.154,-135.795,-770.791,0.011,0.005
5,B,avg_all_signals,long_short,-0.287,0.763,-0.032,-28.380,-721.832,0.018,0.005
9,B,select_by_ic,long_short,0.263,0.278,-0.418,-304.380,-734.038,0.083,0.007
8,B,dynamic_linear_coeff,long_short,0.121,-1.110,-0.396,-548.408,"-1,477.989",-0.309,0.012


## 4. Standalone Alpha Sleeve Benchmark

The alpha sleeves are tested one at a time in both requested signal-set definitions so the final blend is explainable:

- EIA ethanol: direct corn demand.
- Macro/export: FX/export pressure and macro risk.
- Weather: crop-belt weather stress.

For Signal A, the single alpha sleeve is inserted into the A-style family map. For Signal B, the B core is combined with one explicit alpha sleeve.


In [5]:
alpha_signals = {
    name: corn_family_signal(members, trading_index)
    for name, members in families_by_set["alpha"].items()
}

def signal_with_single_alpha(signal_set, alpha_name):
    if signal_set == "A":
        family_map = {
            "prices": dict(families_by_set["A"]["prices"]),
            "fundamentals": dict(families_by_set["B"]["fundamentals"]),
        }
        if alpha_name in ["eia", "weather"]:
            family_map["fundamentals"].update(families_by_set["alpha"][alpha_name])
        elif alpha_name == "macro":
            family_map["macro"] = dict(families_by_set["alpha"]["macro"])
        return "single_alpha_in_core", corn_equal_family_signal(family_map, trading_index)

    b_core = corn_equal_family_signal(families_by_set["B"], trading_index)
    return "core_plus_single_alpha_sleeve", mean_product_flow_signals([b_core, alpha_signals[alpha_name]], trading_index)

alpha_rows = []
alpha_positions = {}
alpha_context_signals = {}
for signal_set in ["A", "B"]:
    for alpha_name in ["eia", "macro", "weather"]:
        strategy, signal = signal_with_single_alpha(signal_set, alpha_name)
        alpha_context_signals[(signal_set, alpha_name)] = signal
        row, _, pos = evaluate_signal("single_alpha_sleeve", signal_set, strategy, signal, note=alpha_name)
        row["alpha"] = alpha_name
        alpha_rows.append(row)
        alpha_positions[(signal_set, alpha_name, "long_short")] = pos

alpha_results = pd.DataFrame(alpha_rows).sort_values(["signal_set", "validation_sharpe"], ascending=[True, False])
display(alpha_results[
    ["signal_set", "alpha", "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "turnover"]
])


,signal_set,alpha,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,turnover
0,A,eia,single_alpha_in_core,long_short,-0.342,0.746,-0.004,-3.436,-618.332,-0.010,0.005
2,A,weather,single_alpha_in_core,long_short,-0.219,0.746,-0.147,-119.854,-735.003,0.015,0.005
1,A,macro,single_alpha_in_core,long_short,-0.226,0.477,0.424,289.445,-318.161,0.112,0.005
3,B,eia,core_plus_single_alpha_sleeve,long_short,-0.416,0.315,0.127,108.109,-405.842,-0.113,0.006
4,B,macro,core_plus_single_alpha_sleeve,long_short,-0.160,0.247,0.656,479.897,-324.074,0.156,0.005
5,B,weather,core_plus_single_alpha_sleeve,long_short,0.105,0.183,-0.423,-192.373,-533.967,-0.035,0.003


## 5. Alpha Combination Candidate Set

Section 4 already tests the standalone EIA, macro/export, and weather sleeves. Here I keep the baseline and only the multi-alpha combinations in the requested Signal A / Signal B structures.

I stop there instead of adding another fixed-weight recipe grid, because the fixed recipes do not feed the final corn choice.


In [6]:
def alpha_combo_names():
    names = ["eia", "macro", "weather"]
    yield tuple()
    for size in range(2, len(names) + 1):
        for combo in combinations(names, size):
            yield combo

combo_rows = []
combo_positions = {}
combo_signals = {}
b_core = corn_equal_family_signal(families_by_set["B"], trading_index)

for combo in alpha_combo_names():
    label = "none" if not combo else "+".join(combo)
    a_map = {
        "prices": dict(families_by_set["A"]["prices"]),
        "fundamentals": dict(families_by_set["B"]["fundamentals"]),
    }
    if "eia" in combo:
        a_map["fundamentals"].update(families_by_set["alpha"]["eia"])
    if "weather" in combo:
        a_map["fundamentals"].update(families_by_set["alpha"]["weather"])
    if "macro" in combo:
        a_map["macro"] = dict(families_by_set["alpha"]["macro"])

    a_signal = corn_equal_family_signal(a_map, trading_index)
    combo_signals[("A", "combo_in_core", label)] = a_signal
    row, _, pos = evaluate_signal("alpha_combo", "A", "combo_in_core", a_signal, note=label)
    row["alpha_combo"] = label
    combo_rows.append(row)
    combo_positions[("A", "combo_in_core", label, "long_short")] = pos

    b_signal = b_core
    if combo:
        alpha_sleeve = mean_product_flow_signals([alpha_signals[name] for name in combo], trading_index)
        b_signal = mean_product_flow_signals([b_core, alpha_sleeve], trading_index)
    combo_signals[("B", "core_plus_alpha_sleeve", label)] = b_signal
    row, _, pos = evaluate_signal("alpha_combo", "B", "core_plus_alpha_sleeve", b_signal, note=label)
    row["alpha_combo"] = label
    combo_rows.append(row)
    combo_positions[("B", "core_plus_alpha_sleeve", label, "long_short")] = pos

combo_results = pd.DataFrame(combo_rows).sort_values(["signal_set", "mode", "validation_sharpe"], ascending=[True, True, False])
display(combo_results[
    ["signal_set", "alpha_combo", "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe"]
])


,signal_set,alpha_combo,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe
0,A,none,combo_in_core,long_short,-0.250,0.787,-0.154,-135.795,-770.791,0.011
4,A,eia+weather,combo_in_core,long_short,-0.309,0.685,-0.005,-3.866,-612.289,-0.009
2,A,eia+macro,combo_in_core,long_short,-0.218,0.415,0.606,399.894,-241.415,0.148
6,A,macro+weather,combo_in_core,long_short,-0.163,0.361,0.570,369.188,-248.134,0.155
8,A,eia+macro+weather,combo_in_core,long_short,-0.164,0.340,0.665,421.742,-231.109,0.174
1,B,none,core_plus_alpha_sleeve,long_short,-0.250,0.787,-0.154,-135.795,-770.791,0.011
3,B,eia+macro,core_plus_alpha_sleeve,long_short,-0.324,0.379,0.488,306.899,-253.611,0.059
7,B,macro+weather,core_plus_alpha_sleeve,long_short,-0.077,0.279,0.326,167.133,-294.422,0.115
9,B,eia+macro+weather,core_plus_alpha_sleeve,long_short,-0.318,0.232,0.377,196.676,-205.550,-0.000
5,B,eia+weather,core_plus_alpha_sleeve,long_short,-0.268,0.192,-0.018,-9.918,-242.837,-0.104


## 6. Final Corn Candidate

These are pre-specified carry-forward candidates from the product-flow research logic, kept because they are economically interpretable and have positive validation performance. OOS is reported only after the candidate is locked.

I lock two plausible base signals, add the product-flow volatility switch as the third candidate, and apply one fixed weak-tape guard. The no-guard rows are kept only as a baseline, not as extra guard variants.


In [7]:
carry_forward_specs = pd.DataFrame([
    {
        "signal_set": "A",
        "promoted_candidate": "combo_in_core / eia+macro+weather",
        "source": "alpha combinations",
        "reason": "Corn-specific prior from product-flow research; combines ethanol, macro/export, and weather; validation positive",
        "source_table": "alpha_combinations",
        "selection_rule": "pre_specified_product_flow_candidate",
        "strategy": "combo_in_core",
        "mode": "long_short",
        "note": "eia+macro+weather",
    },
    {
        "signal_set": "B",
        "promoted_candidate": "core_plus_single_alpha_sleeve / macro",
        "source": "standalone alpha sleeves",
        "reason": "Best B-style single alpha sleeve to carry forward; validation positive; keeps B structure simple",
        "source_table": "standalone_alpha_sleeves",
        "selection_rule": "pre_specified_product_flow_candidate",
        "strategy": "core_plus_single_alpha_sleeve",
        "mode": "long_short",
        "note": "macro",
    },
])
display(Markdown("### Pre-specified carry-forward candidates"))
display(carry_forward_specs[["signal_set", "promoted_candidate", "source", "reason"]])

selected_guard_candidates = build_corn_carry_forward_candidates(
    carry_forward_specs.to_dict("records"),
    combo_results,
    combo_positions,
    combo_signals,
    alpha_results,
    alpha_positions,
    alpha_context_signals,
)

guard_candidate_summary = summarize_corn_candidates(selected_guard_candidates, futures_pnl)
display(Markdown("### Locked base candidates"))
display(guard_candidate_summary[
    ["signal_set", "source_table", "selection_rule", "strategy", "mode", "note", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe"]
])

vol_signal, vol_selected_table, vol_signal_ics, vol_candidate_tables = build_corn_vol_regime_signal(
    signals,
    feature_panels,
    futures_pnl,
)
vol_switch_candidate = make_corn_candidate(
    "product_flow_volatility_switch",
    "ic_family_by_vol_regime",
    "vol_switch",
    "base_regime_ic_vol",
    "long_short",
    "low_normal_family_switch_high_flat",
    vol_signal,
    corn_positions_from_signal(vol_signal, futures_pnl, mode="long_short"),
)
all_guard_candidates = selected_guard_candidates + [vol_switch_candidate]

vol_switch_display = vol_selected_table.drop(columns=["mode"], errors="ignore").assign(backtest_mode="long_short")
display(Markdown("### Product-flow volatility switch decision"))
display(vol_switch_display[["regime", "candidate", "families", "validation_ic", "test_ic", "backtest_mode"]])

supply_masks = corn_abundant_supply_masks(data, feature_panels, futures_pnl)
fixed_guard_specs = [
    {"name": "below_ma_or_negative_mom_flat", "mask": "below_ma_or_negative_mom", "scale": 0.0},
]
supply_results, supply_backtests, supply_positions, _ = run_corn_supply_guard_tests(
    all_guard_candidates,
    supply_masks,
    futures_pnl,
    trading_index,
    oos_start=OOS_START,
    guard_specs=fixed_guard_specs,
)

guard_display_columns = [
    "source_table", "signal_set", "base_strategy", "base_mode", "note", "guard",
    "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover", "guard_oos_pct",
]
guard_summary = supply_results.sort_values(["signal_set", "guard"]).copy()

display(Markdown("### Fixed guard check"))
display(guard_summary[guard_display_columns])


### Pre-specified carry-forward candidates

,signal_set,promoted_candidate,source,reason
0,A,combo_in_core / eia+macro+weather,alpha combinations,Corn-specific prior from product-flow research...
1,B,core_plus_single_alpha_sleeve / macro,standalone alpha sleeves,Best B-style single alpha sleeve to carry forw...


### Locked base candidates

,signal_set,source_table,selection_rule,strategy,mode,note,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe
0,A,alpha_combinations,pre_specified_product_flow_candidate,combo_in_core,long_short,eia+macro+weather,0.340,0.665,421.742,-231.109,0.174
1,B,standalone_alpha_sleeves,pre_specified_product_flow_candidate,core_plus_single_alpha_sleeve,long_short,macro,0.247,0.656,479.897,-324.074,0.156


### Product-flow volatility switch decision

,regime,candidate,families,validation_ic,test_ic,backtest_mode
0,low_vol,selected_all_equal,"price,physical,fx_export,weather,macro",-0.022,0.030,long_short
0,normal_vol,selected_all_equal,"price,fx_export,macro",0.038,0.007,long_short


### Fixed guard check

,source_table,signal_set,base_strategy,base_mode,note,guard,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover,guard_oos_pct
1,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,below_ma_or_negative_mom_flat,-0.530,0.699,128.454,-235.318,0.237,-357.362,0.006,0.780
0,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,no_guard,0.340,0.665,421.742,-231.109,0.174,-741.523,0.004,0.000
3,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,below_ma_or_negative_mom_flat,-0.642,0.623,114.995,-269.137,0.219,-416.039,0.006,0.780
2,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,no_guard,0.247,0.656,479.897,-324.074,0.156,-912.565,0.005,0.000
5,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,-3.834,1.700,304.981,-174.800,0.682,-310.019,0.008,0.780
4,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,no_guard,-1.271,0.670,240.516,-273.055,0.215,-524.684,0.005,0.000


## 7. Final Check

The final row is the best result from the locked candidate set after applying the one fixed guard.


In [8]:
final_row = supply_results.iloc[0]
display(Markdown("### Final row"))
display(pd.DataFrame([final_row])[
    ["source_table", "signal_set", "base_strategy", "base_mode", "note", "guard", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_pnl", "full_dd", "turnover", "guard_oos_pct"]
])

display(
    Markdown(
        f'''
### Conclusion

**Signal A base:** `eia+macro+weather` from alpha combinations.

**Signal B base:** `core_plus_single_alpha_sleeve` / `macro` from standalone alpha sleeves.

**Volatility switch:** product-flow IC family switch by low/normal/high corn volatility.

**Fixed guard:** `{final_row["guard"]}`.

**Final corn-only strategy:** `{final_row["base_strategy"]}` from `{final_row["source_table"]}`. OOS Sharpe `{final_row["oos_sharpe"]:.3f}`, OOS PnL `{final_row["oos_pnl"]:.3f}`, OOS DD `{final_row["oos_dd"]:.3f}`, full-period Sharpe `{final_row["full_sharpe"]:.3f}`.

The visible research path is now limited to the locked candidate set, one volatility-switch decision, and one fixed guard.
'''
    )
)


### Final row

,source_table,signal_set,base_strategy,base_mode,note,guard,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_pnl,full_dd,turnover,guard_oos_pct
5,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,1.700,304.981,-174.800,0.682,277.714,-310.019,0.008,0.780



### Conclusion

**Signal A base:** `eia+macro+weather` from alpha combinations.

**Signal B base:** `core_plus_single_alpha_sleeve` / `macro` from standalone alpha sleeves.

**Volatility switch:** product-flow IC family switch by low/normal/high corn volatility.

**Fixed guard:** `below_ma_or_negative_mom_flat`.

**Final corn-only strategy:** `base_regime_ic_vol` from `product_flow_volatility_switch`. OOS Sharpe `1.700`, OOS PnL `304.981`, OOS DD `-174.800`, full-period Sharpe `0.682`.

The visible research path is now limited to the locked candidate set, one volatility-switch decision, and one fixed guard.
